# **TASK A**





In [ ]:
#Mount Google Drive to access the dataset file
from google.colab import drive
drive.mount('/content/drive')

#Set the file path to the dataset in Google Drive
file_path = "/content/drive/MyDrive/ML/Food Delivery Dataset.csv"

Import all the neccessary libraries for data analysis

In [ ]:
import pandas as pd  #For data manipulation and analysis
import numpy as np   #For numerical operations
import matplotlib.pyplot as plt  #For creating visualisations
import seaborn as sns  #For enhanced visualisations
from sklearn.preprocessing import StandardScaler  #To scale features
from sklearn.cluster import KMeans  #Clustering algorithm
from sklearn.decomposition import PCA  #For dimensionality reduction
from sklearn.metrics import silhouette_score  #to evaluate clustering quality
import warnings  #To handle warning messages

Loading the dataset

In [ ]:
#Load the CSV file from Google Drive into the pandas DataFrame
df = pd.read_csv(file_path)
#Display basic information about the dataset
print("DATASET OVERVIEW")
print(f"Dataset contains {df.shape[0]} rows and {df.shape[1]} columns")
print(f"\n Column names: {list(df.columns)}")

#Check data types of each column
print("\n DATA TYPES:")
print(df.dtypes)

#Check for missing values in each column
print("\n MISSING VALUES PER COLUMN:")
missing_values = df.isnull().sum()
print(missing_values[missing_values > 0])

#Displaying the first 5 rows to show the data structure
print("\nFIRST 5 ROWS OF DATA:")
print(df.head())

Data cleaning and preparation

In [ ]:
#Creating a copy of the original data to preserve it
df_clean = df.copy()

print("DATA CLEANING PROCESS")

#Filling the missing numerical values with the median
numerical_columns = ['total_items', 'subtotal', 'min_item_price', 'max_item_price','total_onshift_partners', 'total_busy_partners', 'total_outstanding_orders']

for column in numerical_columns:
    if column in df_clean.columns:
        median_value = df_clean[column].median()
        df_clean[column] = df_clean[column].fillna(median_value)
        print(f"Filled missing values in {column} with median: {median_value:.2f}")

#Filling missing categorical values with the mode
categorical_columns = ['market_id', 'store_primary_category', 'order_protocol']

for column in categorical_columns:
    if column in df_clean.columns:
        mode_value = df_clean[column].mode()[0]
        df_clean[column] = df_clean[column].fillna(mode_value)
        print(f"Filled missing values in {column} with mode: {mode_value}")

#Convert complaint column to binary (1 for YES, 0 for NO/empty)
df_clean['complaint_binary'] = df_clean['complaint'].apply(lambda x: 1 if x == 'YES' else 0)

#Convert time columns to datetime format for time calculations
df_clean['created_at'] = pd.to_datetime(df_clean['created_at'], format='%H:%M:%S', errors='coerce')
df_clean['actual_delivery_time'] = pd.to_datetime(df_clean['actual_delivery_time'], format='%H:%M:%S', errors='coerce')

#Calculate delivery duration in minutes
df_clean['delivery_duration_minutes'] = (
    (df_clean['actual_delivery_time'] - df_clean['created_at']).dt.total_seconds() / 60
)

#Remove unrealistic delivery times (negative or extremely long)
#Keep only deliveries between 1 minute and 3 hours
df_clean = df_clean[(df_clean['delivery_duration_minutes'] > 0) & (df_clean['delivery_duration_minutes'] < 180)]

print(f"\nAfter cleaning, dataset has {df_clean.shape[0]} rows and {df_clean.shape[1]} columns")


EDA (Exploratory Data Analysis)



In [ ]:

print("EDA")
#Display basic statistics for numerical columns
print("\nBASIC STATISTICS FOR NUMERICAL COLUMNS:")
print(df_clean[numerical_columns].describe())

#First visualisation: Distribution of Categorical Variables
#Create a figure with 2 rows and 2 columns for visualisations
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('Distribution of Categorical Variables', fontsize=16, fontweight='bold')

#Plot 1: Distribution of markets
market_counts = df_clean['market_id'].value_counts()
axes[0, 0].bar(market_counts.index, market_counts.values, color='skyblue', edgecolor='black')
axes[0, 0].set_title('Distribution of Markets')
axes[0, 0].set_xlabel('Market Area')
axes[0, 0].set_ylabel('Number of Orders')
axes[0, 0].tick_params(axis='x', rotation=45)

#Plot 2: Top 10 store categories
top_categories = df_clean['store_primary_category'].value_counts().head(10)
axes[0, 1].barh(top_categories.index, top_categories.values, color='lightgreen', edgecolor='black')
axes[0, 1].set_title('Top 10 Store Categories')
axes[0, 1].set_xlabel('Number of Orders')

#Plot 3: Distribution of order protocols
protocol_counts = df_clean['order_protocol'].value_counts()
axes[1, 0].bar(protocol_counts.index, protocol_counts.values, color='lightcoral', edgecolor='black')
axes[1, 0].set_title('Distribution of Order Protocols')
axes[1, 0].set_xlabel('Protocol Type')
axes[1, 0].set_ylabel('Number of Orders')

#Plot 4: Complaint distribution
complaint_counts = df_clean['complaint_binary'].value_counts()
axes[1, 1].pie(complaint_counts.values, labels=['No Complaint', 'Complaint'], autopct='%1.1f%%', colors=['orange', 'lightcoral'],startangle=90)
axes[1, 1].set_title('Complaint Distribution')
plt.tight_layout()
plt.show()

#Second visualisation: Distribution of Numerical Variables
# Create a 2x4 grid for numerical variable histograms
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
fig.suptitle('Distribution of Numerical Variables', fontsize=16, fontweight='bold')
# Plot histograms for each numerical column
for idx, column in enumerate(numerical_columns[:8]):
    row = idx // 4  # Calculate row position (0 or 1)
    col = idx % 4   # Calculate column position (0-3)
    # Create histogram for current column
    axes[row, col].hist(df_clean[column].dropna(), bins=30, color='red', edgecolor='black', alpha=0.7)
    axes[row, col].set_title(f'{column} Distribution')
    axes[row, col].set_xlabel(column)
    axes[row, col].set_ylabel('Frequency')
plt.tight_layout()
plt.show()

#Third visualisation: Delivery Duration Analysis
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
#Plot 1: Histogram of delivery duration
axes[0].hist(df_clean['delivery_duration_minutes'], bins=30, color='cornflowerblue', edgecolor='black', alpha=0.7)
axes[0].set_title('Distribution of Delivery Duration')
axes[0].set_xlabel('Delivery Duration (minutes)')
axes[0].set_ylabel('Number of Orders')
axes[0].grid(True, alpha=0.3)
#Plot 2: Boxplot of delivery duration by market
#Group data by market for boxplot
market_groups = []
market_labels = []
for market in df_clean['market_id'].unique():
    market_data = df_clean[df_clean['market_id'] == market]['delivery_duration_minutes']
    market_groups.append(market_data)
    market_labels.append(market)

axes[1].boxplot(market_groups, labels=market_labels)
axes[1].set_title('Delivery Duration by Market Area')
axes[1].set_xlabel('Market Area')
axes[1].set_ylabel('Delivery Duration (minutes)')
axes[1].tick_params(axis='x', rotation=45)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

#Fourth visualisation: Correlation Matrix
print("\nCORRELATION ANALYSIS BETWEEN NUMERICAL FEATURES:")
#Select numerical features for correlation analysis
correlation_features = ['total_items', 'subtotal', 'min_item_price', 'max_item_price','total_onshift_partners', 'total_busy_partners', 'total_outstanding_orders', 'delivery_duration_minutes','complaint_binary']
# Calculate correlation matrix
correlation_matrix = df_clean[correlation_features].corr()
# Create heatmap visualisation
plt.figure(figsize=(12, 8))
sns.heatmap(correlation_matrix,
            annot=True,
            cmap='coolwarm',
            center=0,
            square=True,
            linewidths=1,
            cbar_kws={"shrink": 0.8})
plt.title('Correlation Matrix of Numerical Features', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

#Fifth visualisation: Relationship between Order Value and Delivery Time
plt.figure(figsize=(12, 6))
#Created scatter plot with colour coding for complaints
scatter_plot = plt.scatter(df_clean['subtotal'],
                          df_clean['delivery_duration_minutes'],
                          c=df_clean['complaint_binary'],  # Colour by complaint status
                          alpha=0.6,
                          cmap='RdYlGn_r',  # Red-Yellow-Green colour map (reversed)
                          edgecolor='black',
                          linewidth=0.5)

plt.xlabel('Order Subtotal (£)', fontsize=12)
plt.ylabel('Delivery Duration (minutes)', fontsize=12)
plt.title('Order Value vs Delivery Duration (Coloured by Complaint Status)',
          fontsize=14, fontweight='bold')
#Add colour bar legend
colour_bar = plt.colorbar(scatter_plot)
colour_bar.set_label('Complaint Status\n(1 = Yes, 0 = No)', rotation=270, labelpad=20)

plt.grid(True, alpha=0.3)
plt.show()

Data preparation for clustering

In [ ]:
print("PREPARING DATA FOR CLUSTERING ANALYSIS")

#Select relevant features for clustering
clustering_features = [
    'total_items',  #Number of items in order
    'subtotal',  #Order value
    'min_item_price',  #Cheapest item price
    'max_item_price',  #Most expensive item price
    'total_onshift_partners',  #Available delivery partners
    'total_busy_partners',  #Busy delivery partners
    'total_outstanding_orders',  #pending orders
    'delivery_duration_minutes'  #Delivery time
]

#Extract features for clustering
X = df_clean[clustering_features].copy()
#Fill any remaining missing values with column median
X = X.fillna(X.median())
#Standardise features this is important for distance-based clustering
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)  #Transform to have mean=0 and std=1

print(f"Prepared {X_scaled.shape[0]} samples with {X_scaled.shape[1]} features for clustering")
print(f"Features used: {clustering_features}")

To determine the optimal number of clusters


In [ ]:
print("DETERMINING OPTIMAL NUMBER OF CLUSTERS")

#Initialise lists to store metrics
inertia_values = []  #Sum of squared distances to nearest cluster centre
silhouette_scores_list = []  #Measure of how similar points are to their own cluster
k_range = range(2, 11)  #Test cluster numbers from 2 to 10

print("Testing different numbers of clusters:")
print("-" * 50)

for k in k_range:
    #Create KMeans model with current k value
    kmeans_model = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans_model.fit(X_scaled)  # Fit model to scaled data

    #Store inertia (within-cluster sum of squares)
    inertia_values.append(kmeans_model.inertia_)

    #Calculate silhouette score (requires at least 2 clusters)
    if k > 1:
        score = silhouette_score(X_scaled, kmeans_model.labels_)
        silhouette_scores_list.append(score)
        print(f"Clusters: {k} | Inertia: {inertia_values[-1]:.2f} | Silhouette Score: {score:.3f}")

#Sixth visualisation: Elbow Method and Silhouette Score
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

#Plot 1: Elbow Method
axes[0].plot(k_range, inertia_values, 'bo-', linewidth=2, markersize=8)
axes[0].set_xlabel('Number of Clusters (k)', fontsize=12)
axes[0].set_ylabel('Inertia (Within-cluster Sum of Squares)', fontsize=12)
axes[0].set_title('Elbow Method for Optimal k', fontsize=14, fontweight='bold')
axes[0].grid(True, alpha=0.3)

#Highlight potential elbow points
axes[0].axvline(x=3, color='red', linestyle='--', alpha=0.5, label='Potential Elbow at k=3')
axes[0].axvline(x=4, color='green', linestyle='--', alpha=0.5, label='Potential Elbow at k=4')
axes[0].legend()

#Plot 2: Silhouette Scores
axes[1].plot(range(2, 11), silhouette_scores_list, 'ro-', linewidth=2, markersize=8)
axes[1].set_xlabel('Number of Clusters (k)', fontsize=12)
axes[1].set_ylabel('Silhouette Score', fontsize=12)
axes[1].set_title('Silhouette Score for Different k Values', fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

#Based on visual inspection, choose optimal number of clusters
optimal_clusters = 4  # Can be adjusted based on elbow plot
print(f"\nSelected optimal number of clusters: {optimal_clusters}")
print("(Based on visual inspection of elbow plot and silhouette scores)")

Apply the K-menas clustering


In [ ]:
print("APPLYING K-MEANS CLUSTERING ALGORITHM")

#Create final K-Means model with optimal clusters
final_kmeans = KMeans(n_clusters=optimal_clusters, random_state=42, n_init=20)
final_kmeans.fit(X_scaled)  #Train model on scaled data

#Assign cluster labels to each order
df_clean['cluster_label'] = final_kmeans.predict(X_scaled)

print("CLUSTER DISTRIBUTION:")
cluster_counts = df_clean['cluster_label'].value_counts().sort_index()
for cluster_id, count in cluster_counts.items():
    percentage = (count / len(df_clean)) * 100
    print(f"Cluster {cluster_id}: {count} orders ({percentage:.1f}%)")



Visualise the clusters using PCA

In [ ]:
print("VISUALISING CLUSTERS WITH PCA (DIMENSIONALITY REDUCTION)")

#Reduce dimensions to 2 for visualisation
pca = PCA(n_components=2)  #Create PCA model for 2D visualisation
X_pca = pca.fit_transform(X_scaled)  #Transform data to 2 principal components

#Add PCA components to dataframe for plotting
df_clean['pca_component_1'] = X_pca[:, 0]
df_clean['pca_component_2'] = X_pca[:, 1]

print(f"PCA Explained Variance:")
print(f"  Component 1: {pca.explained_variance_ratio_[0]:.1%}")
print(f"  Component 2: {pca.explained_variance_ratio_[1]:.1%}")
print(f"  Total Variance Explained: {sum(pca.explained_variance_ratio_):.1%}")

#Seventh visualisation: Cluster Visualisation with PCA

plt.figure(figsize=(12, 8))
# Scatter plot of clusters in PCA space
scatter = plt.scatter(df_clean['pca_component_1'],
                     df_clean['pca_component_2'],
                     c=df_clean['cluster_label'],  #Colour by cluster
                     cmap='tab10',  #Colour map with distinct colours
                     alpha=0.6,  #Slightly transparent points
                     s=50,  #Point size
                     edgecolor='black',  #Black outline
                     linewidth=0.5)  #Thin outline

plt.xlabel(f'PCA Component 1 ({pca.explained_variance_ratio_[0]:.1%} variance)', fontsize=12)
plt.ylabel(f'PCA Component 2 ({pca.explained_variance_ratio_[1]:.1%} variance)', fontsize=12)
plt.title('Order Clusters Visualised in 2D Space (PCA)', fontsize=16, fontweight='bold')

#Add cluster centres to plot
cluster_centres_pca = pca.transform(final_kmeans.cluster_centers_)
plt.scatter(cluster_centres_pca[:, 0],
           cluster_centres_pca[:, 1],
           c='red',  #Red colour for centres
           marker='X',  #X marker for centres
           s=200,  #Large size
           label='Cluster Centres',  #Legend label
           edgecolor='black',  #Black outline
           linewidth=2)  #Thick outline

plt.legend()  #Show legend
plt.grid(True, alpha=0.3)  #Add light grid
plt.tight_layout()
plt.show()

Analyse cluster characteristics


In [ ]:
print("ANALYSING CLUSTER CHARACTERISTICS")

#Create list to store summary statistics for each cluster
cluster_summaries = []

#Analyse each cluster
for cluster_id in range(optimal_clusters):
    #Get data for current cluster
    cluster_data = df_clean[df_clean['cluster_label'] == cluster_id]
    #Calculate summary statistics
    summary = {
        'cluster_id': cluster_id,
        'order_count': len(cluster_data),
        'percentage_of_total': (len(cluster_data) / len(df_clean)) * 100,
        'avg_items': cluster_data['total_items'].mean(),
        'avg_subtotal': cluster_data['subtotal'].mean(),
        'avg_delivery_time': cluster_data['delivery_duration_minutes'].mean(),
        'complaint_rate': cluster_data['complaint_binary'].mean() * 100,
        'most_common_market': cluster_data['market_id'].mode()[0],
        'most_common_category': cluster_data['store_primary_category'].mode()[0]
    }
    cluster_summaries.append(summary)

#Convert summary list to DataFrame for better display
summary_df = pd.DataFrame(cluster_summaries)

print("\nCLUSTER SUMMARY STATISTICS:")
print("-" * 80)
print(summary_df.to_string(index=False, float_format=lambda x: f"{x:.2f}"))

#Visualisation of the Cluster Characteristics Comparison
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle('Cluster Characteristics Comparison', fontsize=16, fontweight='bold')

#Plot 1: Number of orders per cluster
axes[0, 0].bar(summary_df['cluster_id'],
              summary_df['order_count'],
              color=['blue', 'green', 'red', 'purple'][:optimal_clusters],
              edgecolor='black')
axes[0, 0].set_title('Number of Orders per Cluster')
axes[0, 0].set_xlabel('Cluster')
axes[0, 0].set_ylabel('Number of Orders')

#Plot 2: Average order value per cluster
axes[0, 1].bar(summary_df['cluster_id'],
              summary_df['avg_subtotal'],
              color=['blue', 'green', 'red', 'purple'][:optimal_clusters],
              edgecolor='black')
axes[0, 1].set_title('Average Order Value per Cluster')
axes[0, 1].set_xlabel('Cluster')
axes[0, 1].set_ylabel('Average Subtotal (£)')

#Plot 3: Average delivery time per cluster
axes[0, 2].bar(summary_df['cluster_id'],
              summary_df['avg_delivery_time'],
              color=['blue', 'green', 'red', 'purple'][:optimal_clusters],
              edgecolor='black')
axes[0, 2].set_title('Average Delivery Time per Cluster')
axes[0, 2].set_xlabel('Cluster')
axes[0, 2].set_ylabel('Average Delivery Time (minutes)')

#Plot 4: Complaint rate per cluster
axes[1, 0].bar(summary_df['cluster_id'],
              summary_df['complaint_rate'],
              color=['blue', 'green', 'red', 'purple'][:optimal_clusters],
              edgecolor='black')
axes[1, 0].set_title('Complaint Rate per Cluster')
axes[1, 0].set_xlabel('Cluster')
axes[1, 0].set_ylabel('Complaint Rate (%)')

#Plot 5: Average number of items per cluster
axes[1, 1].bar(summary_df['cluster_id'],
              summary_df['avg_items'],
              color=['blue', 'green', 'red', 'purple'][:optimal_clusters],
              edgecolor='black')
axes[1, 1].set_title('Average Number of Items per Cluster')
axes[1, 1].set_xlabel('Cluster')
axes[1, 1].set_ylabel('Average Items per Order')

#Plot 6: Most common store categories per cluster
for i, row in summary_df.iterrows():
    axes[1, 2].text(0.1, 0.8 - i*0.15,
                   f"Cluster {row['cluster_id']}: {row['most_common_category'][:20]}",
                   fontsize=10,
                   transform=axes[1, 2].transAxes)
axes[1, 2].set_title('Most Common Store Category per Cluster')
axes[1, 2].set_xlim(0, 1)
axes[1, 2].set_ylim(0, 1)
axes[1, 2].axis('off')  #Turn off axis for text display

plt.tight_layout()
plt.show()


Identify the representative orders for each cluster

In [ ]:
print("IDENTIFYING REPRESENTATIVE ORDERS FOR EACH CLUSTER")

#Reset index to make things simpler
df_reset = df_clean.reset_index(drop=True)

#Find the order closest to each cluster centre (most representative)
representative_order_indices = []

for cluster_id in range(optimal_clusters):
    #Get row indices for this cluster (0-based)
    cluster_indices = df_reset.index[df_reset['cluster_label'] == cluster_id].tolist()

    #Get scaled data for current cluster
    cluster_data_scaled = X_scaled[cluster_indices]

    #Get cluster centre coordinates
    cluster_centre = final_kmeans.cluster_centers_[cluster_id]

    #Calculate Euclidean distance from each point to cluster centre
    distances = np.sqrt(np.sum((cluster_data_scaled - cluster_centre) ** 2, axis=1))

    #Find index of point with minimum distance (most representative)
    closest_point_index = np.argmin(distances)
    original_index = cluster_indices[closest_point_index]
    representative_order_indices.append(original_index)

    #Get representative order details
    representative_order = df_reset.loc[original_index]

    print(f"\nREPRESENTATIVE ORDER FOR CLUSTER {cluster_id}:")
    print("-" * 50)
    print(f"Market Area: {representative_order['market_id']}")
    print(f"Store Category: {representative_order['store_primary_category']}")
    print(f"Order Protocol: {representative_order['order_protocol']}")
    print(f"Number of Items: {representative_order['total_items']}")
    print(f"Order Value: £{representative_order['subtotal']:.2f}")
    print(f"Delivery Time: {representative_order['delivery_duration_minutes']:.1f} minutes")
    print(f"Complaint Filed: {representative_order['complaint']}")
    print(f"Order Day: {representative_order['created_day']}")

    #Handle the time formatting safely
    order_time = representative_order['created_at']
    if pd.isna(order_time):
        time_str = "Unknown"
    else:
        time_str = order_time.strftime('%H:%M')
    print(f"Order Time: {time_str}")

#Display all representative orders in a table
print("\n" + "=" * 60)
print("SUMMARY OF REPRESENTATIVE ORDERS")
print("=" * 60)

representative_orders = df_reset.loc[representative_order_indices]

display_columns = [
    'market_id',
    'store_primary_category',
    'order_protocol',
    'total_items',
    'subtotal',
    'delivery_duration_minutes',
    'complaint',
    'created_day'
]

print(representative_orders[display_columns].to_string(index=False))

Save results

In [ ]:
#Save the clustered dataset
output_file_path = "/content/drive/MyDrive/ML/food_delivery_data_with_clusters.csv"
df_clean.to_csv(output_file_path, index=False)